In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
#imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df.drop(columns=['Order_ID'])

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df.isnull().sum())

In [ ]:
#fill categorical missing values with none
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    df[col] = df[col].fillna('none')
# Fill numerical with mode
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0])
df['Delivery_Time'] = df['Delivery_Time'].fillna(df['Delivery_Time'].mode()[0])
print("Missing values remaining:", df.isnull().sum().sum())

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Encode categorical columns - converts text to integers
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le =  LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

In [ ]:
# Task 5: Write your code here:
# Define features (X) and target (y)


features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")


In [ ]:
#Target seems to be balanced

In [ ]:
# Task 1: Write your code here:
# Define features (X) and target (y)
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df['Delivery_Time'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5  # K=5 Folds

# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model = RandomForestClassifier(n_estimators=100, max_depth=15,
                               class_weight='balanced', random_state=42)
  model.fit(X_train, y_train)

print("Model trained!")

In [ ]:
# Task 1: Write your code here:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
df[y_test].hist(bins=30, edgecolor='black')

plt.title(f"Target Distribution ({y_test})")
plt.xlabel(y)
plt.ylabel("Frequency")
plt.grid(False)

plt.show()



In [ ]:
# Task Bonus: Write your code here: